In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Flipkart Pipeline").getOrCreate()

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]


In [0]:
df=spark.createDataFrame(data,columns)

In [0]:
df.write.format("delta").mode("append").save("/Volumes/workspace/default/my_volume")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6095679999596999>, line 1
----> 1 df.write.format("delta").mode("append").save("/Volumes/workspace/default/my_volume")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_in_plan(req.plan, command)
-> 1538 data, _, metri

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/my_volume")

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknowm,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import col,to_date
df=df.withColumn("amount",col("amount").cast("int"))\
    .withColumn("date",to_date("date"))

In [0]:
df=df.dropna(subset=["amount"])

In [0]:
df=df.fillna({"city":"unknowm"})

In [0]:
df=df.filter(col("amount")>0)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknowm,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
window_spec=Window.partitionBy("order_id").orderBy(col("date").desc())

In [0]:
df=df.withColumn("rn",row_number().over(window_spec)).filter(col("rn")==1).drop("rn")
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknowm,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import initcap
df=df.withColumn("city",initcap(col("city")))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/my_volume")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/my_volume")
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknowm,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import sum,count
sales_product=df.groupBy("product").agg(sum("amount").alias("total_sales"))
sales_product.display()

product,total_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


In [0]:
sales_category=df.groupBy("category").agg(sum("amount").alias("total_sales"))
sales_category.display()

category,total_sales
Electronics,160000
Accessories,11000


In [0]:
sales_city=df.groupBy("city").agg(sum("amount").alias("sales_city"))
sales_city.display()

city,sales_city
Delhi,55000
Unknowm,3000
Chennai,33000
Hyderabad,72000
Mumbai,8000


In [0]:
count_order=df.groupBy("customer_id").agg(count("order_id")).alias("total_order")
count_order.display()

customer_id,count(order_id)
C005,1
C004,1
C003,1
C001,2
C002,1
C007,1


In [0]:
total_spend=df.groupBy("customer_id").agg(sum("amount").alias("total_spend"))
total_spend.display()

customer_id,total_spend
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


In [0]:
top_product=df.groupBy("product").agg(sum("quantity").alias("top_product")).orderBy("top_product",ascending=False)
top_product.display()


product,top_product
Mobile,3
Laptop,2
Tablet,1
Watch,1
Headphones,1


In [0]:
top_customer=df.groupBy("customer_id").agg(sum("amount").alias("top_custome")).orderBy("top_custome",ascending=False)
top_customer.display()

customer_id,top_custome
C001,72000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000
